# Movie Genre Classification – EDA & Results

CodSoft ML Internship · Task 1

This notebook walks through the analysis using the reusable modules in `src/`. The heavy lifting (tuning, refitting, final evaluation) is done by `python train.py`; this notebook reads its saved reports.

In [ ]:
import sys, json
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))
import pandas as pd
from IPython.display import Image, display
from src.data_loader import load_train_data, load_test_data
from src.eda import dataset_summary
from src.preprocessing import clean_text, build_text

## 1. Load and inspect the dataset

In [ ]:
train, train_report = load_train_data()
test, test_report = load_test_data()
print(train_report.as_dict())
print(test_report.as_dict())
train.head()

## 2. Missing values, duplicates and imbalance

In [ ]:
summary = dataset_summary(train, test)
{k: v for k, v in summary.items() if k not in ('genre_counts', 'median_words_by_genre', 'description_words')}

In [ ]:
pd.Series(summary['genre_counts']).to_frame('movies').assign(share=lambda d: (d.movies / d.movies.sum()).round(4))

In [ ]:
display(Image(str(ROOT / 'reports/figures/genre_distribution.png')))

## 3. Plot description lengths

In [ ]:
print(pd.Series(summary['description_words']).round(1))
display(Image(str(ROOT / 'reports/figures/description_length_distribution.png')))
display(Image(str(ROOT / 'reports/figures/description_length_by_genre.png')))

## 4. Common and distinctive words by genre

In [ ]:
display(Image(str(ROOT / 'reports/figures/common_words_by_genre.png')))
display(Image(str(ROOT / 'reports/figures/top_words_by_genre.png')))

## 5. Text cleaning example

In [ ]:
row = train.iloc[0]
print('RAW :', row.title, '|', row.description[:200])
print('CLEAN:', clean_text(build_text(row.title, row.description))[:200])

## 6. Model comparison

Models were tuned and **selected on a stratified 20% validation split** of `train_data.txt` (macro-F1). The official test labels were used only once afterwards.

In [ ]:
pd.read_csv(ROOT / 'reports/validation_model_comparison.csv')

In [ ]:
pd.read_csv(ROOT / 'reports/test_model_comparison.csv')

In [ ]:
display(Image(str(ROOT / 'reports/figures/model_comparison_test.png')))

## 7. Selected model – per-genre results and confusion matrix

In [ ]:
pd.read_csv(ROOT / 'reports/per_genre_metrics_test.csv')

In [ ]:
display(Image(str(ROOT / 'reports/figures/confusion_matrix_test.png')))

## 8. Using the saved pipeline

In [ ]:
from src.predictor import GenrePredictor
p = GenrePredictor.load()
p.predict('A retired gunslinger defends the ranchers of a frontier town from a gang of outlaws.', title='Dust and Iron')